In [ ]:
# � LEVEL 4: Dataset Handling
# 👉 Goal: Prepare real training data


In [6]:
# 32. Load images from a folder into a list
import os
from PIL import Image
def load_images_from_folder(folder_path):
    images = []
    for filename in os.listdir(folder_path):
        if filename.endswith((".jpg", ".png",".jpeg")):
            img_path = os.path.join(folder_path, filename)
            img = Image.open(img_path).convert('RGB')
            images.append(img)
    return images
# img_path = "../"
# my_images = load_images_from_folder(img_path)
# print(f"loaded {len(my_images)} images")
##-------- os.listdir(), os.path.join()

loaded 0 images


In [ ]:

# 33. Assign labels (real = 0, AI = 1)
import os

def load_data_with_labels(real_folder, ai_folder):
    data = []
    labels = []

    # Process Real Images
    for filename in os.listdir(real_folder):
        if filename.endswith((".jpg", ".png")):
            data.append(os.path.join(real_folder, filename))
            labels.append(0)  # 0 represents 'Real'

    # Process AI Images
    for filename in os.listdir(ai_folder):
        if filename.endswith((".jpg", ".png")):
            data.append(os.path.join(ai_folder, filename))
            labels.append(1)  # 1 represents 'AI'

    return data, labels

# Usage
# file_paths, targets = load_data_with_labels("data/real", "data/ai")

In [ ]:
# 34. Create a dataset dictionary structure
# Assuming 'file_paths' and 'targets' are the lists from step 33
dataset = {
    "image_path": file_paths,
    "label": targets,
    "metadata": {
        "class_names": ["Real", "AI"],
        "total_count": len(file_paths)
    }
}

# Accessing data is now very intuitive:
print(f"Path: {dataset['image_path'][0]}")
print(f"Label: {dataset['label'][0]} ({dataset['metadata']['class_names'][dataset['label'][0]]})")

In [ ]:
# 35. Read dataset paths from CSV file
import pandas as pd

# 1. Load the CSV file
df = pd.read_csv("dataset.csv")

# 2. View the first few rows (optional)
print(df.head())

# 3. Convert columns into lists (back to our Step 34 format!)
file_paths = df['filepath'].tolist()
labels = df['label'].tolist()

# Now you have your lists ready for training!

##------------- pd.read_csv(), .head(), .tolist()

In [ ]:
# 36. Shuffle dataset
import pandas as pd

# Load your data
df = pd.read_csv("dataset.csv")

# Shuffle the rows
# 'frac=1' means shuffle 100% of the data
# 'random_state=42' is a "seed" so you get the same shuffle every time you run it
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(df_shuffled.head())

In [ ]:
# 37. Split into train/test (80/20)
from sklearn.model_selection import train_test_split

# Let's say 'df_shuffled' is our shuffled DataFrame from the last step
train_df, test_df = train_test_split(
    df_shuffled, 
    test_size=0.20,    # 20% for testing
    random_state=42    # Keeps the split consistent
)

print(f"Training samples: {len(train_df)}")
print(f"Testing samples: {len(test_df)}")

In [ ]:
# 38. Write a custom dataset class (basic)
from torch.utils.data import Dataset
from PIL import Image

class ImageDataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = file_paths
        self.labels = labels
        self.transform = transform # For later (resizing, etc.)

    def __len__(self):
        # Return the total number of images
        return len(self.file_paths)

    def __getitem__(self, index):
        # 1. Get the path and label for the requested index
        img_path = self.file_paths[index]
        label = self.labels[index]
        
        # 2. Load the actual image
        image = Image.open(img_path).convert("RGB")
        
        # 3. Apply changes (if any)
        if self.transform:
            image = self.transform(image)
            
        return image, label

# How to use it:
# my_train_dataset = ImageDataset(train_paths, train_labels)